# DER parsing: tag, length, value
Run `02_openssl_keys.ipynb` first – it creates the key files used here. Run cells from top to bottom. Modify only cells marked `TODO`; the check cells after them verify your implementation.

In Part 1 you complete a small DER parser and compare it with `openssl asn1parse`. In Part 2 you decode the same files with the Python `asn1` library.

In [ ]:
import os

for name in ["RSA_2048_priv.der", "RSA_2048_pub.der", "EC_256_priv.der"]:
    assert os.path.exists(name), f"{name} is missing - run 02_openssl_keys.ipynb first"

## Part 1: Manual DER parser
This is a teaching parser for the RSA and EC DER files used here. It implements only the DER features these examples need
(single-byte tags, a few universal types, definite lengths) and does no error checking – it is not a complete ASN.1 implementation.

### Task 1: Length
Implement `length(data_bytes, offset)`. It returns the decoded length and the offset of the first byte after the length:
- short form: one byte `< 0x80` is the length itself;
- long form: `0x80 | k` – the top bit is set and the lower 7 bits give the number `k` of length bytes that follow (big endian);
- `0x80` alone means indefinite length, which is not allowed in DER.

The parser `tlv` below prints every TLV with its offset, tag and length; it descends into `SEQUENCE`, `SET` and unknown (e.g. context-specific) tags.

In [ ]:
types_verbose = \
{
 0x02:'INTEGER',
 0x03:'BIT STRING',
 0x04:'OCTET STRING',
 0x05:'NULL',
 0x06:'OBJECT',
 0x13:'PRINTABLESTRING',
 0x14:'T61STRING',
 0x17:'UTCTIME',
 0x30:'SEQUENCE',
 0x31:'SET'
 }

def tag(data_bytes, offset):
    ID = data_bytes[offset]
    if ID in types_verbose:
        return types_verbose[ID], offset + 1
    else:
        return ID, offset + 1

def length(data_bytes, offset):
    # TODO: decode the short form (< 128) and the long form (first byte is 0x80 | number_of_length_bytes)
    return length, offset # returns length and new offset


def value(data_bytes, offset,  length):
    return data_bytes[offset: offset + length], offset + length

def tlv(data_bytes, offset, depth=0):
    offset_backup = offset  # position where tlv starts
    t, offset = tag(data_bytes, offset)
    l, offset = length(data_bytes, offset)

    if (t in ['SEQUENCE', 'SET']) or not(t in types_verbose.values()):
        print('\t' * depth, f'offset={offset_backup}, tag={t}, length={l}')
        sequence_end = offset + l
        while (offset < sequence_end):
            offset = tlv(data_bytes, offset, depth + 1)

    else:
        v, offset = value(data_bytes, offset, l)
        print('\t' * depth, f'offset={offset_backup}, tag={t}, length={l}')
    return offset

In [ ]:
assert length(bytes([0x05]), 0) == (5, 1)
assert length(bytes([0x81, 0xff]), 0) == (255, 2)
assert length(bytes([0x82, 0x01, 0x00]), 0) == (256, 3)
print("length works")

### Task 2: Parse real files
Run the parser on `RSA_2048_pub.der` and compare its output with `asn1parse`. Then run it on the RSA and EC private keys;
the checks verify that the parser consumes each complete DER structure.
The parser treats `BIT STRING` as a primitive value and therefore does not descend into the DER structure stored inside it.
Complete the TODO cell to parse that nested structure (skip the `BIT STRING` header and the unused-bits byte).

In [ ]:
bytes_der = open('RSA_2048_pub.der', 'rb').read()
assert tlv(bytes_der, 0, depth=0) == len(bytes_der)
!openssl asn1parse -i -inform DER -in RSA_2048_pub.der

In [ ]:
end = tlv(bytes_der, ..., depth=1)  # TODO: replace ... by the offset of the SEQUENCE inside the BIT STRING
assert end == len(bytes_der)

In [ ]:
for name in ['RSA_2048_priv.der', 'EC_256_priv.der']:
    der = open(name, 'rb').read()
    assert tlv(der, 0, depth=0) == len(der)

## Part 2: The `asn1` library
The helper code below is adapted from the example [`dump.py`](https://github.com/andrivet/python-asn1/blob/master/examples/dump.py) of the python-asn1 library
(MIT license, copyright (c) 2007-2016 by the Python-ASN1 authors).
Documentation of `asn1`: https://python-asn1.readthedocs.io/en/latest/reference/asn1.html

In [ ]:
import sys
import binascii

import asn1


tag_id_to_string_map = {
    asn1.Numbers.Boolean: "BOOLEAN",
    asn1.Numbers.Integer: "INTEGER",
    asn1.Numbers.BitString: "BIT STRING",
    asn1.Numbers.OctetString: "OCTET STRING",
    asn1.Numbers.Null: "NULL",
    asn1.Numbers.ObjectIdentifier: "OBJECT",
    asn1.Numbers.PrintableString: "PRINTABLESTRING",
    asn1.Numbers.IA5String: "IA5STRING",
    asn1.Numbers.UTCTime: "UTCTIME",
    asn1.Numbers.GeneralizedTime: "GENERALIZED TIME",
    asn1.Numbers.Enumerated: "ENUMERATED",
    asn1.Numbers.Sequence: "SEQUENCE",
    asn1.Numbers.Set: "SET"
}

class_id_to_string_map = {
    asn1.Classes.Universal: "U",
    asn1.Classes.Application: "A",
    asn1.Classes.Context: "C",
    asn1.Classes.Private: "P"
}

object_id_to_string_map = {
    "1.2.840.113549.1.1.1": "rsaEncryption",
    "1.2.840.113549.1.1.5": "sha1WithRSAEncryption",

    "1.3.6.1.5.5.7.1.1": "authorityInfoAccess",

    "2.5.4.3": "commonName",
    "2.5.4.4": "surname",
    "2.5.4.5": "serialNumber",
    "2.5.4.6": "countryName",
    "2.5.4.7": "localityName",
    "2.5.4.8": "stateOrProvinceName",
    "2.5.4.9": "streetAddress",
    "2.5.4.10": "organizationName",
    "2.5.4.11": "organizationalUnitName",
    "2.5.4.12": "title",
    "2.5.4.13": "description",
    "2.5.4.42": "givenName",

    "1.2.840.113549.1.9.1": "emailAddress",

    "2.5.29.14": "X509v3 Subject Key Identifier",
    "2.5.29.15": "X509v3 Key Usage",
    "2.5.29.16": "X509v3 Private Key Usage Period",
    "2.5.29.17": "X509v3 Subject Alternative Name",
    "2.5.29.18": "X509v3 Issuer Alternative Name",
    "2.5.29.19": "X509v3 Basic Constraints",
    "2.5.29.30": "X509v3 Name Constraints",
    "2.5.29.31": "X509v3 CRL Distribution Points",
    "2.5.29.32": "X509v3 Certificate Policies Extension",
    "2.5.29.33": "X509v3 Policy Mappings",
    "2.5.29.35": "X509v3 Authority Key Identifier",
    "2.5.29.36": "X509v3 Policy Constraints",
    "2.5.29.37": "X509v3 Extended Key Usage"
}


def tag_id_to_string(identifier):
    """Return a string representation of a ASN.1 id."""
    if identifier in tag_id_to_string_map:
        return tag_id_to_string_map[identifier]
    return '{:#02x}'.format(identifier)


def class_id_to_string(identifier):
    """Return a string representation of an ASN.1 class."""
    if identifier in class_id_to_string_map:
        return class_id_to_string_map[identifier]
    raise ValueError('Illegal class: {:#02x}'.format(identifier))


def object_identifier_to_string(identifier):
    if identifier in object_id_to_string_map:
        return object_id_to_string_map[identifier]
    return identifier


def value_to_string(tag_number, value):
    if tag_number == asn1.Numbers.ObjectIdentifier:
        return object_identifier_to_string(value)
    elif isinstance(value, bytes):
        return '0x' + str(binascii.hexlify(value).upper())
    elif isinstance(value, str):
        return value
    else:
        return repr(value)


def pretty_print(input_stream, output_stream, indent=0):
    """Pretty print ASN.1 data."""
    while not input_stream.eof():
        tag = input_stream.peek()
        if tag.typ == asn1.Types.Primitive:
            tag, value = input_stream.read()
            output_stream.write(' ' * indent)
            output_stream.write('[{}] {}: {}\n'.format(class_id_to_string(tag.cls), tag_id_to_string(tag.nr), value_to_string(tag.nr, value)))
        elif tag.typ == asn1.Types.Constructed:
            output_stream.write(' ' * indent)
            output_stream.write('[{}] {}\n'.format(class_id_to_string(tag.cls), tag_id_to_string(tag.nr)))
            input_stream.enter()
            pretty_print(input_stream, output_stream, indent + 2)
            input_stream.leave()

In [ ]:
decoder = asn1.Decoder()
decoder.start(open('RSA_2048_priv.der', 'rb').read())
pretty_print(decoder, sys.stdout)

### Task 3: Extract values
Based on `pretty_print` above, implement `der_to_values`, which appends all primitive ASN.1 values – already decoded to Python values such as `int` – to a list.
The check compares the extracted RSA numbers with the values loaded by the `cryptography` library.

In [ ]:
def der_to_values(input_stream, res):
    """Append primitive ASN.1 values to res while recursively walking the tree."""
    pass  # TODO

In [ ]:
from cryptography.hazmat.primitives import serialization

decoder = asn1.Decoder()
encoded_bytes = open('RSA_2048_priv.der', 'rb').read()
decoder.start(encoded_bytes)

values = []
der_to_values(decoder, values)
print(*values, sep='\n')

numbers = serialization.load_der_private_key(encoded_bytes, None).private_numbers()
assert values[1:6] == [numbers.public_numbers.n, numbers.public_numbers.e, numbers.d, numbers.p, numbers.q]
print("values match")